# 02 - Dataset Deep Dive and Split Strategy

Primary dataset: `Clinton/Text-to-sql-v1`.

Goal: understand structure, complexity, limitations, and robust split design.


## What Is This Technique?

### What is this technique?
Schema-aware dataset curation and stratified splitting

### Definition and core concepts.
Transform raw instruction-input-response rows into structured training examples with parsed schema metadata and complexity tags.

### Why was this technique developed?
Mixed-source SQL datasets can hide distribution skew and complexity imbalance, causing misleading model quality signals.

### What limitations of traditional RAG does it solve?
Traditional RAG evaluation rarely addresses data split leakage for executable query tasks. This curation pipeline enforces source+complexity-aware split behavior.

### Architecture and workflow diagram explanation.
Raw rows -> schema parsing -> SQL normalization -> complexity tagging -> stratified train/val/test split -> persisted manifests.

### Component-by-component breakdown.
Schema parser, normalization utilities, complexity heuristics, stratified splitter, manifest writer.

### When should it be used in real-world systems?
Use for any text-to-query fine-tuning setup where schema distribution varies across domains.

### Advantages and disadvantages.
Advantages: reliable evaluation and better generalization diagnostics. Disadvantages: extra preprocessing and edge-case handling for malformed schemas.

### Comparison against standard RAG and other implemented RAG variants.
Compared with random splits, stratified splits reduce variance and leakage risk. Compared with document-RAG corpora, query datasets require executable-target consistency checks.

### Implementation details and design decisions used in this project.
We parse CREATE TABLE blocks directly from dataset context and preserve source labels for downstream error slicing.


In [ ]:
from datasets import load_dataset

ds = load_dataset("Clinton/Text-to-sql-v1", split="train")
print(ds)
print(ds.features)
print(ds[0])


In [ ]:
import re
from collections import Counter
import numpy as np

sources = Counter(ds["source"])
print("Top sources:", sources.most_common(10))

sqls = ds["response"]
patterns = {
    "join": r"\bjoin\b",
    "group_by": r"\bgroup\s+by\b",
    "nested": r"\(\s*select\b",
}
for name, pat in patterns.items():
    c = sum(1 for q in sqls if re.search(pat, q, flags=re.I))
    print(name, c)

lengths = [len(x) for x in sqls]
print("p95 chars", np.percentile(lengths, 95))


In [ ]:
import pandas as pd
from repo_query_gen.data_prep import run_data_preparation

paths = run_data_preparation("fast")
train_df = pd.read_csv(paths["train_csv"])
val_df = pd.read_csv(paths["val_csv"])
test_df = pd.read_csv(paths["test_csv"])

print(len(train_df), len(val_df), len(test_df))
print(train_df[["source", "complexity_bucket"]].head())


## Post-Run Analysis (dataset curation and split strategy)

After executing the real pipeline, use this section to document measured outcomes.

- Analyze actual outputs, metrics, retrieval quality, latency, and generation quality.
- Explain how the technique changed measured behavior vs baseline.
- Interpret every metric in business and engineering terms.
- Capture failure modes, lessons learned, and practical takeaways.
- Explain why specific outputs were produced and what they demonstrate.
- Conclude effectiveness based on measured evidence, not assumptions.
